# GPU probe: `probe` on a CUDA device

## Question and why it matters
Can the growth/shape `probe` (width, area, edge, gyration windows per ant) run on the GPU for thousands of (rule, seed) runs, with output **byte-identical** to the CPU `probe`, and how much faster is it? Links: `README.md`, `NOTEBOOK.md`.

## Assumptions and definitions
- CPU `probe` is the oracle; the GPU must equal it byte for byte (tested, held).
- Speedup = CPU whole-process wall / GPU whole-process wall, same 2400 specs, `probe 27 8192`, 6 CPU threads vs one RTX 4000 Ada at 2400 lanes.
- Assumed: machine idle during timing (checked by process list / load average in `bench.log`).

## Provenance
Prints the git commit, date, seed and a SHA-256 for every input file.

In [1]:
import hashlib, subprocess, datetime, json, re
from pathlib import Path
SEED = 0  # no randomness in this notebook: it only reads committed results
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
EXP = ROOT / "experiments/active/2026-10-gpu-probe"
R = EXP / "results"
inputs = [R/"bench.log", R/"bench_gpu1.json", R/"bench_gpu2.json", R/"spotcheck.txt", R/"spotcheck_specs.txt", R/"bench_lanes.log"] + [R/f"bench_lanes_{L}_r{k}.json" for L in (1024, 1536, 2400) for k in (1, 2)]
missing = [p for p in inputs if not p.exists()]
assert not missing, missing
print("notebook run", datetime.date.today(), "seed", SEED)
print("HEAD", subprocess.run(["git","-C",str(ROOT),"rev-parse","--short","HEAD"],capture_output=True,text=True).stdout.strip())
for p in inputs: print(hashlib.sha256(p.read_bytes()).hexdigest()[:16], p.relative_to(ROOT))

notebook run 2026-10-19 seed 0
HEAD 0169fee
1ff7b06d99ef84cc experiments/2026-10-16-gpu-probe/results/bench.log
cd692d366597c82c experiments/2026-10-16-gpu-probe/results/bench_gpu1.json
42713bbb43909ad6 experiments/2026-10-16-gpu-probe/results/bench_gpu2.json
254196fcde6ce8fc experiments/2026-10-16-gpu-probe/results/spotcheck.txt
088ea3b969c2352e experiments/2026-10-16-gpu-probe/results/spotcheck_specs.txt
1b50cfee957766c3 experiments/2026-10-16-gpu-probe/results/bench_lanes.log
16898b7f1aacd600 experiments/2026-10-16-gpu-probe/results/bench_lanes_1024_r1.json
0c48c44010c5316a experiments/2026-10-16-gpu-probe/results/bench_lanes_1024_r2.json
5bcefaeac7e6291f experiments/2026-10-16-gpu-probe/results/bench_lanes_1536_r1.json
3e72a933e617bc1e experiments/2026-10-16-gpu-probe/results/bench_lanes_1536_r2.json
a3969565a4154fdd experiments/2026-10-16-gpu-probe/results/bench_lanes_2400_r1.json
1a1ac5d3160c00ee experiments/2026-10-16-gpu-probe/results/bench_lanes_2400_r2.json


## Data and method
`bash experiments/active/2026-10-gpu-probe/scripts/bench.sh` (WSL, repo root) times, sequentially, `gpu_probe 27 8192 --lanes 2400` (gpu1), `probe 27 8192` (cpu), and gpu2 on the 2400 kappa-leadrun-fresh specs. Heavy work is in the Rust binaries; this notebook parses `bench.log`.

In [2]:
log = (R/"bench.log").read_text()
wall = {m.group(1): float(m.group(2)) for m in re.finditer(r"(gpu1|cpu|gpu2) wall ([\d.]+) s", log)}
ident = re.findall(r"cmp vs dump: (\w+)", log)
print(wall, ident)
print(log.splitlines()[0])
for k in ("gpu1","gpu2"):
    s = json.loads((R/f"bench_{k}.json").read_text()); print(k, s)
print("speedup gpu1: %.2fx  gpu2: %.2fx" % (wall["cpu"]/wall["gpu1"], wall["cpu"]/wall["gpu2"]))

{'gpu1': 177.355451168, 'cpu': 681.648633111, 'gpu2': 176.949274154} ['identical', 'identical', 'identical']
git 53ccccc dirty=60 2026-10-17T17:38:27+13:00
gpu1 {'jobs': 2400, 'wall_secs': 160.9595, 'device': 'NVIDIA RTX 4000 Ada Generation', 'lanes': 2400, 'grid': 8192, 'slice': 1048576, 'max_log2': 27, 'pool_tiles': 2409257, 'launches': 128, 'oom_requeues': 0}


gpu2 {'jobs': 2400, 'wall_secs': 161.2787, 'device': 'NVIDIA RTX 4000 Ada Generation', 'lanes': 2400, 'grid': 8192, 'slice': 1048576, 'max_log2': 27, 'pool_tiles': 2409257, 'launches': 128, 'oom_requeues': 0}
speedup gpu1: 3.84x  gpu2: 3.85x


**Superseded 2026-10-19 (see "P2 re-measure" below): this 3.84x is from a stale binary; at HEAD P2 is falsified from above (5.81x, CPU baseline shift unexplained).**

## Results
GPU 177.4 s / 176.9 s vs CPU 681.6 s: **3.84x**, all outputs byte-identical to the existing CPU dump (`oom_requeues` 0, pool 2.41e6 tiles).

*Shows:* the GPU port is exact on this workload and about 3.8x faster, inside the pre-registered 2x-6x (P2 held; P1, P3 held). *Does not show:* a general speed claim; n = 1 CPU run, one spec set, one GPU; lane scaling (P4) was not run.

## Caveats and controls
- The first GPU figure (177 s, 20:58 on 10-16) and the old CPU time (1231 s) were contaminated by other jobs; the clean CPU time is 681.6 s. A handover estimate of ~7x used the contaminated number and is superseded.
- Registered pool estimate (1.6e6 tiles) was about 50% low; no requeues.
- 24-spec spot check: `spotcheck.txt` (byte-identical).

## P4: throughput vs lanes (added 2026-10-18)
`scripts/bench_lanes.sh` runs `gpu_probe 27 8192 --lanes L` on the same 2400 specs for L = 1024, 1536, 2400, two repeats each, interleaved, with a freshly built HEAD binary (sha256 91c072f7...). Registered prediction: steps/s rises with lanes from 1024 to 2400. Criterion fixed before running (NOTEBOOK.md): mean time strictly falls 1024 > 1536 > 2400, each gap larger than the larger repeat spread, same order on the device path. Steps/s uses one numerator (`total_steps`, identical across L because outputs are).

In [3]:
import statistics as st
llog = (R/"bench_lanes.log").read_text()
print(llog.splitlines()[1])
assert len(re.findall(r"cmp vs dump: identical", llog)) == 6, "not all phases byte-identical"
assert all(l.strip() == "foreign procs:" for l in llog.splitlines() if "foreign procs" in l), "contaminated phase"
wallp = {(int(m.group(1)), int(m.group(2))): float(m.group(3)) for m in re.finditer(r"lanes_(\d+)_r(\d) exit 0 wall ([\d.]+) s", llog)}
rows = {}
for L in (1024, 1536, 2400):
    s = [json.loads((R/f"bench_lanes_{L}_r{k}.json").read_text()) for k in (1, 2)]
    assert len({x["total_steps"] for x in s}) == 1 and s[0]["oom_requeues"] == 0
    w = [wallp[(L, k)] for k in (1, 2)]; d = [x["wall_secs"] for x in s]
    rows[L] = dict(wall=st.mean(w), wspread=max(w)-min(w), dev=st.mean(d), dspread=max(d)-min(d), steps=s[0]["total_steps"], launches=s[0]["launches"])
print("lanes  wall(mean, spread)   device(mean, spread)   steps/s(dev)  per-lane  launches")
for L, r in rows.items():
    sps = r["steps"]/r["dev"]
    print(f"{L:5d}  {r['wall']:7.2f} {r['wspread']:5.2f}       {r['dev']:7.2f} {r['dspread']:5.2f}        {sps:.3e}   {sps/L:.3e}  {r['launches']}")
Ls = (1024, 1536, 2400)
def holds(key, spr):
    big = max(rows[L][spr] for L in Ls)
    return all(rows[a][key] - rows[b][key] > big for a, b in zip(Ls, Ls[1:]))
print("P4 held:", holds("wall", "wspread") and holds("dev", "dspread"))

gpu_probe sha256 91c072f715cf5da6b5e2aaf380551d188d8c7d1058d60a2968cfb9c14b3653d3 mtime 2026-10-18 19:57:05.460035117 +1300
lanes  wall(mean, spread)   device(mean, spread)   steps/s(dev)  per-lane  launches
 1024   280.30  0.10        278.13  0.02        9.820e+08   9.590e+05  319
 1536   219.35  0.12        217.21  0.10        1.257e+09   8.186e+05  256
 2400   156.44  0.40        154.26  0.19        1.771e+09   7.378e+05  128
P4 held: True


*Shows:* P4 held. Batch steps/s rises 1.28x (1024 to 1536) and 1.41x (1536 to 2400), and the repeats agree to within 0.4 s. All six outputs are byte-identical to the CPU dump, the first full-scale identity check of the HEAD binary.

*Does not show:* lane-level parallel efficiency. Below 2400 lanes, ants queue (about 2.3 and 1.6 job-rounds), so most of the rise is just queueing. Per-lane steps/s actually falls 23% from 1024 to 2400 lanes; the cause is untested. Single spec set, n = 2 per L, one GPU.

*Provenance caveat:* the P2 timing above used a stale binary built 2026-10-16 20:46, not 53ccccc. Cargo did not rebuild because the source mtimes on `/mnt/f` predate later edits (README, NOTEBOOK 2026-10-18). Do not divide P4 times into P2's.

## P2 re-measure on a verified HEAD build (added 2026-10-19)
`scripts/bench_p2.sh`: fresh target dir, build exit status checked, binaries verified newer than sources, sha256 logged; gpu1, cpu, gpu2 sequential on the same 2400 specs. Prediction (registered 2026-10-16, not edited): 2x-6x, falsified at >= 4.9x or < 2x.

In [4]:
plog = (R/"bench_p2.log").read_text()
pw = {m.group(1): float(m.group(2)) for m in re.finditer(r"(gpu1|cpu|gpu2) exit 0 wall ([\d.]+) s", plog)}
assert len(re.findall(r"cmp vs dump: identical", plog)) == 3 and "cmp cpu vs gpu1: identical" in plog
print(plog.splitlines()[0]); print(plog.splitlines()[2]); print(plog.splitlines()[3])
print(pw)
print(f"speedup gpu1: {pw['cpu']/pw['gpu1']:.2f}x  gpu2: {pw['cpu']/pw['gpu2']:.2f}x; registered falsification line 4.9x")
print(json.loads((R/"bench_p2_gpu1.json").read_text())["wall_secs"], json.loads((R/"bench_p2_gpu2.json").read_text())["wall_secs"])


git 6de71f2 dirty(src,tests,Cargo.toml; filemode ignored)=0 2026-10-19T08:32:09+13:00
probe sha256 7d341949a912bc14a7b1c83f72a4fab14d41e6868e3717b7a6bfc19ecb190a29 mtime 2026-10-19 08:31:55.946175713 +1300
gpu_probe sha256 91c072f715cf5da6b5e2aaf380551d188d8c7d1058d60a2968cfb9c14b3653d3 mtime 2026-10-19 08:31:56.050175429 +1300
{'gpu1': 157.056047201, 'cpu': 912.48550584, 'gpu2': 156.938488521}
speedup gpu1: 5.81x  gpu2: 5.81x; registered falsification line 4.9x
154.7854 154.7216


*Shows:* all outputs byte-identical; 912.5 s / 157.0 s = **5.81x**, so P2 is falsified at HEAD from above (>= 4.9x). *Does not show:* why. GPU time moved little (177 -> 157 s); the CPU baseline rose from 681.6 s to 912.5 s (+34%) for an unexplained reason (the old CPU binary came from the suspect target dir; the `src/probe.rs` refactor and machine state are candidates, not bisected). n = 1 CPU run, so quote the speedup as roughly 4-6x, not 5.8x. Tests on this build: `cargo test --release` ok; `--features cuda --test gpu --test onset --test gpu_probe` 7/7, 1/1, 5/5.

## What this changes, next steps, open questions
Use `gpu_probe` for batches of hundreds+ plain-rule runs, with lanes >= number of jobs when they fit (P4). P2 was rerun on a verified build (5.81x, falsified from above). Next: find why the HEAD CPU probe is 34% slower than the old one; find why per-lane throughput drops with occupancy; optional one-byte packing for n <= 32 if VRAM binds. Build hygiene: on `/mnt/f`, do not trust cargo's freshness check for timed binaries; log and check the sha.